In [4]:
import os
os.chdir("..")
import pandas as pd

df = pd.read_csv("data/diary.csv")
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

print(df.shape)
print(df.columns.tolist())
print(df.head())

(588, 8)
['date', 'name', 'year', 'letterboxd_uri', 'rating', 'rewatch', 'tags', 'watched_date']
         date                                               name  year  \
0  2022-07-29                                  Chungking Express  1994   
1  2022-07-29                                        Fire Island  2022   
2  2022-07-29  The French Dispatch of the Liberty, Kansas Eve...  2021   
3  2022-07-30                                  Dog Day Afternoon  1975   
4  2022-07-30                                     Loving Vincent  2017   

           letterboxd_uri  rating rewatch tags watched_date  
0  https://boxd.it/34Dund     5.0     NaN  NaN   2021-02-15  
1  https://boxd.it/34D9Av     4.0     NaN  NaN   2022-07-28  
2  https://boxd.it/34CWDB     4.0     NaN  NaN   2022-07-29  
3  https://boxd.it/34J8Tj     4.5     NaN  NaN   2022-07-29  
4  https://boxd.it/34Kypr     3.5     NaN  NaN   2022-07-29  


In [11]:
print("Total films:", len(df))
print()
print("Missing ratings:", df["rating"].isna().sum())
print("Missing watched_date:", df["watched_date"].isna().sum())
print()
print("Rewatches:", df["rewatch"].notna().sum())
print("Entries with tags:", df["tags"].notna().sum())
print()
print("Rating distribution:")
print(df["rating"].value_counts().sort_index())

Total films: 588

Missing ratings: 0
Missing watched_date: 0

Rewatches: 165
Entries with tags: 454

Rating distribution:
rating
0.5     16
1.0      4
1.5     14
2.0     37
2.5     42
3.0     61
3.5     73
4.0    163
4.5     58
5.0    120
Name: count, dtype: int64


In [17]:
tags_series = df["tags"].dropna().str.split(",").explode().str.strip()
print("Unique tags:", tags_series.nunique())
print()
print("Most common tags:")
print(tags_series.value_counts().head(20))

Unique tags: 42

Most common tags:
tags
w/ ria               244
w/ irene              98
brother nefarious     90
criterion             66
watchers              65
w/ nadia              52
screening             33
theater               29
class                 23
plane watch           17
w/ sarah              15
w/ alex               14
w/ nick               13
w/ gibby              13
w/ v                  12
fall break            11
spring break          10
w/ finch               9
brother malicious      6
w/ nana-nani           5
Name: count, dtype: int64


In [22]:
def categorize_tags(tag_str):
    if pd.isna(tag_str):
        return []
    tags = [t.strip().lower() for t in tag_str.split(",")]
    categories = []
    for tag in tags:
        if tag.startswith("w/"):
            categories.append("social")
        elif tag in ["theater", "theatre", "screening", "imax"]:
            categories.append("theater")
        elif tag in ["criterion"]:
            categories.append("criterion")
        elif tag in ["class"]:
            categories.append("class")
        elif "break" in tag or "plane" in tag:
            categories.append("travel_or_break")
        elif "brother" in tag:
            categories.append("background")
    return categories

df["tag_categories"] = df["tags"].apply(categorize_tags)

print("Social watches:", df["tag_categories"].apply(lambda x: "social" in x).sum())
print("Theater watches:", df["tag_categories"].apply(lambda x: "theater" in x).sum())
print("Criterion watches:", df["tag_categories"].apply(lambda x: "criterion" in x).sum())
print("Background watches:", df["tag_categories"].apply(lambda x: "background" in x).sum())
print("Class watches:", df["tag_categories"].apply(lambda x: "class" in x).sum())
print("Travel/break:", df["tag_categories"].apply(lambda x: "travel_or_break" in x).sum())
bg_mask = df["tag_categories"].apply(lambda x: "background" in x)
print("Average rating for background watches:", df[bg_mask]["rating"].mean().round(2))
print("Average rating for everything else:", df[~bg_mask]["rating"].mean().round(2))

Social watches: 371
Theater watches: 61
Criterion watches: 66
Background watches: 90
Class watches: 23
Travel/break: 36
Average rating for background watches: 2.43
Average rating for everything else: 3.91


In [26]:
theater_mask = df["tag_categories"].apply(lambda x: "theater" in x)
criterion_mask = df["tag_categories"].apply(lambda x: "criterion" in x)
rewatch_mask = df["rewatch"] == "Yes"

print("Average rating for theater:", round(df[theater_mask]["rating"].mean(), 2))
print("Average rating for criterion:", round(df[criterion_mask]["rating"].mean(), 2))
print("Average rating for rewatches:", round(df[rewatch_mask]["rating"].mean(), 2))
print("Average rating for all:", round(df["rating"].mean(), 2))

Average rating for theater: 3.66
Average rating for criterion: 4.3
Average rating for rewatches: 4.03
Average rating for all: 3.68


In [29]:
df["watched_date"] = pd.to_datetime(df["watched_date"])
df["year_watched"] = df["watched_date"].dt.year
df["month_watched"] = df["watched_date"].dt.month

print("Watches by year:")
print(df["year_watched"].value_counts().sort_index())
print()
print("Watches by month (all years):")
print(df["month_watched"].value_counts().sort_index())

Watches by year:
year_watched
2021      1
2022     28
2023    183
2024    179
2025    142
2026     55
Name: count, dtype: int64

Watches by month (all years):
month_watched
1     48
2     51
3     49
4     60
5     65
6     35
7     40
8     29
9     45
10    62
11    44
12    60
Name: count, dtype: int64


In [31]:
oct_films = df[df["month_watched"] == 10][["name", "year", "rating"]].sort_values("year")
print(f"Total October watches: {len(oct_films)}")
print()
print(oct_films.to_string(index=False))

Total October watches: 62

                                              name  year  rating
                       The Cabinet of Dr. Caligari  1920     3.5
                                         Nosferatu  1922     3.5
                                           Persona  1966     5.0
                                   Rosemary's Baby  1968     4.0
                                   Mikey and Nicky  1976     4.5
                                         Halloween  1978     3.5
                                          Superman  1978     3.0
                                          Scanners  1981     3.5
                                        Possession  1981     5.0
                    An American Werewolf in London  1981     3.0
                              Possibly in Michigan  1983     5.0
                                        Videodrome  1983     4.5
                                           The Fly  1986     4.0
                                       The Witches  1990     4.

# dataset summary 
# 588 films, 2021–2026 (active logging from late 2022)
# no missing ratings or watched dates
# 
# rating distribution:
#   overall mean:    3.68
#   criterion:       4.30  (makes sense)
#   rewatches:       4.03
#   theater:         3.66
#   background:      2.43 
#
# seasonal pattern:
#   peak months:     april, may, october
#   low months:      june, july, august (summer dip)
#   october:         horror exploration
#
# tag categories:
#   social:          371 watches
#   background:      90  (exclude or downweight in model)
#   criterion:       66
#   theater:         61
#   travel/break:    36
#   class:           23
#
# key modeling decisions:
#   - exclude/downweight background watches (2.43 avg vs 3.91)
#   - treat rewatches as a positive signal